In [1]:
"""
03_model_evaluation.py — Comprehensive model evaluation.

1. Results comparison table (all models x all metrics)
2. Confusion matrices for all models
3. ROC curves (one-vs-rest) for top 3 models
4. Calibration plots for top 3 models
5. Per-class precision/recall/F1 (focus on draw class)
6. Test set evaluation (holdout: 2022 WC onward)
7. 2022 World Cup specific analysis
8. Time-series cross-validation
"""

'\n03_model_evaluation.py — Comprehensive model evaluation.\n\n1. Results comparison table (all models x all metrics)\n2. Confusion matrices for all models\n3. ROC curves (one-vs-rest) for top 3 models\n4. Calibration plots for top 3 models\n5. Per-class precision/recall/F1 (focus on draw class)\n6. Test set evaluation (holdout: 2022 WC onward)\n7. 2022 World Cup specific analysis\n8. Time-series cross-validation\n'

In [2]:
import sys
from pathlib import Path

In [3]:
BASE = (Path.cwd() if (Path.cwd() / 'data' / 'processed').exists() else Path.cwd().parent)
sys.path.insert(0, str(BASE))

In [4]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, f1_score, accuracy_score,
                              log_loss, roc_auc_score)
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

In [5]:
from src.models import evaluate_model

In [6]:
# ── Setup ──────────────────────────────────────────────────────────────
SPLITS = BASE / "data" / "processed" / "splits"
ARTIFACTS = BASE / "outputs" / "model_artifacts"
FIG_DIR = BASE / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
    "figure.facecolor": "white",
})
sns.set_style("whitegrid")

In [8]:
CLASS_NAMES = ["Home Win", "Draw", "Away Win"]
CLASS_COLORS = ["#2a9d8f", "#e9c46a", "#e63946"]

In [9]:
# ── Load data ──────────────────────────────────────────────────────────
print("Loading data and models...")
X_val = pd.read_csv(SPLITS / "X_val.csv")
y_val = pd.read_csv(SPLITS / "y_val.csv")["result"].values
X_test = pd.read_csv(SPLITS / "X_test.csv")
y_test = pd.read_csv(SPLITS / "y_test.csv")["result"].values
X_val_med = pd.read_csv(SPLITS / "X_val_median.csv")
X_test_med = pd.read_csv(SPLITS / "X_test_median.csv")

Loading data and models...


In [10]:
results_df = pd.read_csv(ARTIFACTS / "results_summary.csv")

In [11]:
# Load key models
models = {}
model_files = list(ARTIFACTS.glob("*.joblib"))
for f in model_files:
    if f.stem in ["optuna_results"]:
        continue
    try:
        models[f.stem] = joblib.load(f)
    except Exception as e:
        print(f"  Warning: couldn't load {f.stem}: {e}")

/Users/rzein/Desktop/Projects/KickCaster/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/rzein/Desktop/Projects/KickCaster/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/rzein/Desktop/Projects/KickCaster/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpic

/Users/rzein/Desktop/Projects/KickCaster/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/rzein/Desktop/Projects/KickCaster/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pickle.py:1760: UserWarning: [22:46:01] WARNING

/Users/rzein/Desktop/Projects/KickCaster/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator CalibratedClassifierCV from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/rzein/Desktop/Projects/KickCaster/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator _BinMapper from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/rzein/Desktop/Projects/KickCaster/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unp

In [12]:
print(f"Loaded {len(models)} models: {list(models.keys())}")

Loaded 36 models: ['KNN_k5', 'LightGBM_tuned', 'KNN_k10', 'XGBoost_tuned_SMOTE', 'LightGBM_cal', 'CatBoost_cal', 'KickCastNet', 'kickcast_net_v3_config', 'LightGBM_tuned_balanced', 'KNN_k20', 'XGBoost_tuned_reduced', 'kickcast_net_config', 'SVM_RBF', 'XGBoost_ultra', 'LogisticRegression', 'XGBoost_untuned', 'CatBoost_ultra', 'XGBoost_cal', 'ultra_config', 'LightGBM_ultra', 'enhanced_optuna_results', 'KickCastNet_v3', 'TwoStage_XGBoost', 'HistGBM_cal', 'CatBoost_tuned_balanced', 'XGBoost_tuned_balanced', 'HistGBM_untuned', 'HistGBM_tuned_SMOTE', 'kickcast_net_v2_config', 'HistGBM_tuned_balanced', 'KNN_k50', 'XGBoost_tuned', 'CalibratedEnsemble', 'CatBoost_tuned', 'HistGBM_tuned', 'ultra_features']


In [13]:
optuna_data = joblib.load(ARTIFACTS / "optuna_results.joblib")

In [14]:
# ══════════════════════════════════════════════════════════════════════
# 1. Results comparison table
# ══════════════════════════════════════════════════════════════════════
print("\n[1/8] Results comparison table...")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print(results_df.sort_values("macro_f1", ascending=False).to_string(index=False, float_format="{:.4f}".format))


[1/8] Results comparison table...
                 model  accuracy  macro_f1  log_loss  acc_class_0  acc_class_1  acc_class_2  auc_roc_ovr
XGBoost_tuned_balanced    0.5744    0.5165    1.0029       0.7358       0.2569       0.5599       0.7096
HistGBM_tuned_balanced    0.5465    0.5090    0.9189       0.6512       0.3087       0.5630       0.7154
   HistGBM_tuned_SMOTE    0.5719    0.5052    0.9735       0.7456       0.2107       0.5721       0.7102
         XGBoost_tuned    0.5899    0.5048    1.0060       0.8141       0.1774       0.5463       0.7131
         HistGBM_tuned    0.6054    0.5033    0.8913       0.8390       0.1294       0.5979       0.7265
      TwoStage_XGBoost    0.5714    0.5018    1.0708       0.7420       0.1978       0.5873       0.7105
       XGBoost_untuned    0.5779    0.4906    0.9303       0.7927       0.1516       0.5615       0.7150
   XGBoost_tuned_SMOTE    0.5645    0.4840    1.0477       0.7918       0.1848       0.4886       0.6966
       HistGBM_untun

In [15]:
# ══════════════════════════════════════════════════════════════════════
# 2. Confusion matrices for key models
# ══════════════════════════════════════════════════════════════════════
print("\n[2/8] Confusion matrices...")


[2/8] Confusion matrices...


In [16]:
key_models = ["XGBoost_tuned", "XGBoost_tuned_balanced", "HistGBM_tuned",
              "HistGBM_tuned_balanced", "StackingEnsemble", "LogisticRegression"]
key_models = [m for m in key_models if m in models]

In [17]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

In [18]:
for i, name in enumerate(key_models[:6]):
    model = models[name]
    # Choose correct X based on model type
    if name in ["StackingEnsemble", "RandomForest"]:
        X_v = X_val_med
    else:
        X_v = X_val
    y_pred = model.predict(X_v)
    cm = confusion_matrix(y_val, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues", ax=axes[i],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cbar_kws={"label": "%"})
    # Add count annotations
    for r in range(3):
        for c in range(3):
            axes[i].text(c + 0.5, r + 0.75, f"(n={cm[r,c]})",
                        ha="center", va="center", fontsize=7, color="gray")
    axes[i].set_ylabel("Actual")
    axes[i].set_xlabel("Predicted")
    axes[i].set_title(name.replace("_", " "), fontsize=10)

In [19]:
for j in range(len(key_models), 6):
    axes[j].set_visible(False)

In [20]:
plt.suptitle("Confusion Matrices (Validation Set)", fontsize=14)
plt.tight_layout()
fig.savefig(FIG_DIR / "10_confusion_matrices.png")
plt.close()
print("  Saved: 10_confusion_matrices.png")

  Saved: 10_confusion_matrices.png


In [21]:
# ══════════════════════════════════════════════════════════════════════
# 3. ROC curves (one-vs-rest) for top 3 models
# ══════════════════════════════════════════════════════════════════════
print("\n[3/8] ROC curves...")


[3/8] ROC curves...


In [22]:
top3 = ["XGBoost_tuned_balanced", "HistGBM_tuned_balanced", "HistGBM_tuned"]
top3 = [m for m in top3 if m in models]

In [23]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

In [24]:
for cls_idx in range(3):
    ax = axes[cls_idx]
    y_binary = (y_val == cls_idx).astype(int)

    for m_name in top3:
        model = models[m_name]
        X_v = X_val_med if m_name in ["StackingEnsemble", "RandomForest"] else X_val
        y_proba = model.predict_proba(X_v)[:, cls_idx]
        fpr, tpr, _ = roc_curve(y_binary, y_proba)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, label=f"{m_name.replace('_', ' ')} (AUC={roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC — {CLASS_NAMES[cls_idx]}")
    ax.legend(fontsize=7, loc="lower right")

In [25]:
plt.suptitle("One-vs-Rest ROC Curves (Validation Set)", fontsize=14)
plt.tight_layout()
fig.savefig(FIG_DIR / "11_roc_curves.png")
plt.close()
print("  Saved: 11_roc_curves.png")

  Saved: 11_roc_curves.png


In [26]:
# ══════════════════════════════════════════════════════════════════════
# 4. Calibration plots for top 3 models
# ══════════════════════════════════════════════════════════════════════
print("\n[4/8] Calibration plots...")


[4/8] Calibration plots...


In [27]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

In [28]:
for cls_idx in range(3):
    ax = axes[cls_idx]
    y_binary = (y_val == cls_idx).astype(int)

    for m_name in top3:
        model = models[m_name]
        X_v = X_val_med if m_name in ["StackingEnsemble", "RandomForest"] else X_val
        y_proba = model.predict_proba(X_v)[:, cls_idx]
        prob_true, prob_pred = calibration_curve(y_binary, y_proba, n_bins=10, strategy="uniform")
        ax.plot(prob_pred, prob_true, "o-", linewidth=2,
                label=m_name.replace("_", " "))

    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfectly calibrated")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title(f"Calibration — {CLASS_NAMES[cls_idx]}")
    ax.legend(fontsize=7)

In [29]:
plt.suptitle("Calibration Plots (Validation Set)", fontsize=14)
plt.tight_layout()
fig.savefig(FIG_DIR / "12_calibration_plots.png")
plt.close()
print("  Saved: 12_calibration_plots.png")

  Saved: 12_calibration_plots.png


In [30]:
# ══════════════════════════════════════════════════════════════════════
# 5. Per-class precision/recall/F1 (draw focus)
# ══════════════════════════════════════════════════════════════════════
print("\n[5/8] Per-class P/R/F1 table...")


[5/8] Per-class P/R/F1 table...


In [31]:
print("\nPer-class metrics for top models on validation set:")
for m_name in key_models:
    model = models[m_name]
    X_v = X_val_med if m_name in ["StackingEnsemble", "RandomForest"] else X_val
    y_pred = model.predict(X_v)
    print(f"\n--- {m_name} ---")
    print(classification_report(y_val, y_pred, target_names=CLASS_NAMES, digits=3))


Per-class metrics for top models on validation set:

--- XGBoost_tuned ---
              precision    recall  f1-score   support

    Home Win      0.655     0.814     0.726      1124
        Draw      0.319     0.177     0.228       541
    Away Win      0.576     0.546     0.561       659

    accuracy                          0.590      2324
   macro avg      0.516     0.513     0.505      2324
weighted avg      0.554     0.590     0.563      2324


--- XGBoost_tuned_balanced ---
              precision    recall  f1-score   support

    Home Win      0.667     0.736     0.700      1124
        Draw      0.315     0.257     0.283       541
    Away Win      0.573     0.560     0.566       659

    accuracy                          0.574      2324
   macro avg      0.519     0.518     0.516      2324
weighted avg      0.559     0.574     0.565      2324


--- HistGBM_tuned ---
              precision    recall  f1-score   support

    Home Win      0.650     0.839     0.733      112

In [32]:
# ══════════════════════════════════════════════════════════════════════
# 6. Test set evaluation (holdout)
# ══════════════════════════════════════════════════════════════════════
print("\n[6/8] Test set evaluation (2022 WC holdout)...")


[6/8] Test set evaluation (2022 WC holdout)...


In [33]:
# Evaluate best models on test set
best_models = ["XGBoost_tuned_balanced", "HistGBM_tuned_balanced", "XGBoost_tuned", "StackingEnsemble"]
best_models = [m for m in best_models if m in models]

In [34]:
test_results = []
for m_name in best_models:
    model = models[m_name]
    X_t = X_test_med if m_name in ["StackingEnsemble", "RandomForest"] else X_test
    metrics = evaluate_model(model, X_t, y_test, model_name=m_name)
    test_results.append(metrics)
    print(f"\n  {m_name}:")
    print(f"    Accuracy: {metrics['accuracy']:.4f}  |  Macro F1: {metrics['macro_f1']:.4f}  |  "
          f"Log Loss: {metrics['log_loss']:.4f}  |  AUC-ROC: {metrics['auc_roc_ovr']:.4f}")
    print(f"    Per-class acc: HW={metrics['acc_class_0']:.3f}  D={metrics['acc_class_1']:.3f}  AW={metrics['acc_class_2']:.3f}")


  XGBoost_tuned_balanced:
    Accuracy: 0.5625  |  Macro F1: 0.5008  |  Log Loss: 1.0308  |  AUC-ROC: 0.7057
    Per-class acc: HW=0.714  D=0.206  AW=0.600

  HistGBM_tuned_balanced:
    Accuracy: 0.5625  |  Macro F1: 0.5133  |  Log Loss: 0.9178  |  AUC-ROC: 0.7243
    Per-class acc: HW=0.683  D=0.258  AW=0.608



  XGBoost_tuned:
    Accuracy: 0.5695  |  Macro F1: 0.4771  |  Log Loss: 1.0370  |  AUC-ROC: 0.7095
    Per-class acc: HW=0.791  D=0.118  AW=0.571


In [35]:
test_df = pd.DataFrame(test_results)
test_df.to_csv(ARTIFACTS / "test_results.csv", index=False)

In [36]:
# Confusion matrix for best model on test set
fig, axes = plt.subplots(1, len(best_models), figsize=(5 * len(best_models), 5))
if len(best_models) == 1:
    axes = [axes]

In [37]:
for i, m_name in enumerate(best_models):
    model = models[m_name]
    X_t = X_test_med if m_name in ["StackingEnsemble", "RandomForest"] else X_test
    y_pred = model.predict(X_t)
    cm = confusion_matrix(y_test, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Oranges", ax=axes[i],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[i].set_ylabel("Actual")
    axes[i].set_xlabel("Predicted")
    axes[i].set_title(f"{m_name.replace('_', ' ')}\n(Test Set)")

In [38]:
plt.suptitle("Test Set Confusion Matrices", fontsize=14)
plt.tight_layout()
fig.savefig(FIG_DIR / "13_test_confusion_matrices.png")
plt.close()
print("  Saved: 13_test_confusion_matrices.png")

  Saved: 13_test_confusion_matrices.png


In [39]:
# ══════════════════════════════════════════════════════════════════════
# 7. 2022 World Cup specific analysis
# ══════════════════════════════════════════════════════════════════════
print("\n[7/8] 2022 World Cup analysis...")


[7/8] 2022 World Cup analysis...


In [40]:
# Load full feature matrix to get 2022 WC matches
fm = pd.read_csv(BASE / "data" / "processed" / "feature_matrix.csv", parse_dates=["date"])
wc22 = fm[(fm["tournament"].str.contains("FIFA World Cup", na=False)) &
          (fm["date"] >= "2022-11-20") & (fm["date"] <= "2022-12-18")].copy()

In [41]:
if len(wc22) > 0:
    print(f"  Found {len(wc22)} 2022 World Cup matches")

    DROP_COLS = ["date", "home_team", "away_team", "home_score", "away_score", "tournament", "result"]
    X_wc22 = wc22.drop(columns=DROP_COLS, errors="ignore")
    y_wc22 = wc22["result"].values

    # Use best model
    best_name = "XGBoost_tuned_balanced"
    if best_name in models:
        model = models[best_name]
        y_pred = model.predict(X_wc22)
        y_proba = model.predict_proba(X_wc22)

        print(f"\n  2022 WC predictions ({best_name}):")
        print(f"    Accuracy: {accuracy_score(y_wc22, y_pred):.4f}")
        print(f"    Macro F1: {f1_score(y_wc22, y_pred, average='macro'):.4f}")

        # Per-match predictions
        print("\n  Match-by-match predictions:")
        correct = 0
        for idx, (_, row) in enumerate(wc22.iterrows()):
            pred = y_pred[idx]
            actual = int(row["result"])
            pred_label = CLASS_NAMES[pred]
            actual_label = CLASS_NAMES[actual]
            match_str = f"  {row['home_team']} vs {row['away_team']}"
            probs_str = f"[HW:{y_proba[idx,0]:.2f} D:{y_proba[idx,1]:.2f} AW:{y_proba[idx,2]:.2f}]"
            check = "OK" if pred == actual else "XX"
            if pred == actual:
                correct += 1
            print(f"    {check} {match_str:<40s} Pred: {pred_label:<10s} Actual: {actual_label:<10s} {probs_str}")

        print(f"\n    Correct: {correct}/{len(wc22)} ({100*correct/len(wc22):.1f}%)")

        # Confusion matrix for WC 2022
        cm = confusion_matrix(y_wc22, y_pred, labels=[0, 1, 2])
        fig, ax = plt.subplots(figsize=(6, 5))
        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Greens", ax=ax,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        for r in range(3):
            for c in range(3):
                ax.text(c + 0.5, r + 0.75, f"(n={cm[r,c]})",
                       ha="center", va="center", fontsize=8, color="gray")
        ax.set_title(f"2022 World Cup Predictions — {best_name.replace('_', ' ')}")
        ax.set_ylabel("Actual")
        ax.set_xlabel("Predicted")
        plt.tight_layout()
        fig.savefig(FIG_DIR / "14_wc2022_confusion_matrix.png")
        plt.close()
        print("  Saved: 14_wc2022_confusion_matrix.png")
else:
    print("  No 2022 WC matches found in feature matrix")

  Found 64 2022 World Cup matches

  2022 WC predictions (XGBoost_tuned_balanced):
    Accuracy: 0.4531
    Macro F1: 0.4038

  Match-by-match predictions:
    OK   Qatar vs Ecuador                       Pred: Away Win   Actual: Away Win   [HW:0.15 D:0.27 AW:0.58]
    XX   United States vs Wales                 Pred: Home Win   Actual: Draw       [HW:0.68 D:0.23 AW:0.09]
    OK   England vs Iran                        Pred: Home Win   Actual: Home Win   [HW:0.52 D:0.40 AW:0.08]
    OK   Senegal vs Netherlands                 Pred: Away Win   Actual: Away Win   [HW:0.04 D:0.30 AW:0.66]
    XX   Argentina vs Saudi Arabia              Pred: Home Win   Actual: Away Win   [HW:0.97 D:0.02 AW:0.01]
    OK   Mexico vs Poland                       Pred: Draw       Actual: Draw       [HW:0.18 D:0.43 AW:0.39]
    XX   Denmark vs Tunisia                     Pred: Home Win   Actual: Draw       [HW:0.50 D:0.30 AW:0.21]
    XX   France vs Australia                    Pred: Draw       Actual: Home Win

  Saved: 14_wc2022_confusion_matrix.png


In [42]:
# ══════════════════════════════════════════════════════════════════════
# 8. Time-series cross-validation
# ══════════════════════════════════════════════════════════════════════
print("\n[8/8] Time-series cross-validation...")


[8/8] Time-series cross-validation...


In [43]:
# Expanding window: train to each WC, test on next cycle
fm_full = pd.read_csv(BASE / "data" / "processed" / "feature_matrix.csv", parse_dates=["date"])
DROP_COLS = ["date", "home_team", "away_team", "home_score", "away_score", "tournament", "result"]

In [44]:
windows = [
    ("Train to 2010", "2004-01-01", "2010-07-11", "2010-07-12", "2014-07-12"),
    ("Train to 2014", "2004-01-01", "2014-07-13", "2014-07-14", "2018-07-15"),
    ("Train to 2018", "2004-01-01", "2018-07-15", "2018-07-16", "2022-12-18"),
    ("Train to 2022", "2004-01-01", "2022-12-18", "2022-12-19", "2026-12-31"),
]

In [45]:
cv_results = []
for name, train_start, train_end, test_start, test_end in windows:
    train = fm_full[(fm_full["date"] >= train_start) & (fm_full["date"] <= train_end)]
    test = fm_full[(fm_full["date"] >= test_start) & (fm_full["date"] <= test_end)]

    if len(test) == 0:
        continue

    X_tr = train.drop(columns=DROP_COLS, errors="ignore")
    y_tr = train["result"].values
    X_te = test.drop(columns=DROP_COLS, errors="ignore")
    y_te = test["result"].values

    # Use XGBoost with best params
    xgb_params = optuna_data["xgboost_best_params"]
    model_cv = XGBClassifier(
        objective="multi:softprob", num_class=3,
        random_state=42, n_jobs=-1, tree_method="hist",
        **xgb_params
    )
    # Balanced weights
    class_counts = np.bincount(y_tr)
    cw = len(y_tr) / (3 * class_counts)
    sw = np.array([cw[y] for y in y_tr])
    model_cv.fit(X_tr, y_tr, sample_weight=sw)

    y_pred = model_cv.predict(X_te)
    y_proba = model_cv.predict_proba(X_te)

    cv_res = {
        "window": name,
        "train_size": len(train),
        "test_size": len(test),
        "accuracy": accuracy_score(y_te, y_pred),
        "macro_f1": f1_score(y_te, y_pred, average="macro"),
        "log_loss": log_loss(y_te, y_proba),
    }
    try:
        cv_res["auc_roc"] = roc_auc_score(y_te, y_proba, multi_class="ovr", average="macro")
    except ValueError:
        cv_res["auc_roc"] = np.nan
    cv_results.append(cv_res)
    print(f"  {name}: acc={cv_res['accuracy']:.4f}, F1={cv_res['macro_f1']:.4f}, "
          f"logloss={cv_res['log_loss']:.4f}, train={cv_res['train_size']}, test={cv_res['test_size']}")

  Train to 2010: acc=0.5304, F1=0.4760, logloss=1.1722, train=6090, test=3965


  Train to 2014: acc=0.5432, F1=0.4850, logloss=1.1127, train=10056, test=3770


  Train to 2018: acc=0.5718, F1=0.5056, logloss=1.0088, train=13826, test=4080


  Train to 2022: acc=0.5677, F1=0.5077, logloss=0.9998, train=17906, test=3465


In [46]:
cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(ARTIFACTS / "timeseries_cv_results.csv", index=False)

In [47]:
# Plot CV metrics over time
if len(cv_results) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    metrics_to_plot = [("accuracy", "Accuracy"), ("macro_f1", "Macro F1"), ("log_loss", "Log Loss")]
    for ax, (col, label) in zip(axes, metrics_to_plot):
        ax.plot(range(len(cv_df)), cv_df[col], "o-", color="#264653", linewidth=2, markersize=8)
        ax.set_xticks(range(len(cv_df)))
        ax.set_xticklabels(cv_df["window"], rotation=30, ha="right")
        ax.set_ylabel(label)
        ax.set_title(f"Time-Series CV: {label}")
        ax.grid(True, alpha=0.3)

    plt.suptitle("Expanding Window Cross-Validation (XGBoost Balanced)", fontsize=14)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "15_timeseries_cv.png")
    plt.close()
    print("  Saved: 15_timeseries_cv.png")

  Saved: 15_timeseries_cv.png


In [48]:
print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)


EVALUATION COMPLETE
